In [9]:
import geopandas as gpd
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
# Load the shapefile containing the coastal profiles
profiles = gpd.read_file(os.path.join("data", "profiles",  "perfiles_playas_limpiosCuadrantes94a120.shp"))
# Create a new column for the profile ID
profiles["profile_id"] = profiles.index + 1

# Load the shorelines
shorelines = gpd.read_file(os.path.join("data", "shorelines",  "LineasCosta.gpkg"),
                           layer="LineasCostaPlayas")
# Keep only the relevant columns
columns_to_keep = ["ID_LineaCosta", "Municipio_norm", "Playa_norm", "Fecha", "Hora", "Cuadrante", "NivelTotal"]
shorelines = shorelines[columns_to_keep + ["geometry"]]
# Filter the shorelines to keep only those where "Cuadrante" is between 94 and 120
shorelines = shorelines[(shorelines["Cuadrante"] >= 94) & (shorelines["Cuadrante"] <= 120)]

# Ensure both GeoDataFrames use the same coordinate reference system (CRS)
profiles = profiles.to_crs(shorelines.crs)
print(f"Profiles CRS: {profiles.crs}")

Profiles CRS: EPSG:25830


In [ ]:
# Intersect profiles and shorelines, retaining only point intersections.
profiles_shoreline_intersections = gpd.overlay(
    profiles[["profile_id", "geometry"]],
    shorelines,
    how="intersection",
    keep_geom_type=False,
).explode(index_parts=False, ignore_index=True)
profiles_shoreline_intersections = profiles_shoreline_intersections[
    profiles_shoreline_intersections.geometry.geom_type == "Point"
].copy()

# Shapely's project() returns the along-profile distance in the CRS units (meters here).
profile_geometries = profiles.set_index("profile_id").geometry
profiles_shoreline_intersections["shoreline_position_m"] = (
    profiles_shoreline_intersections.apply(
        lambda row: profile_geometries.loc[row["profile_id"]].project(row.geometry),
        axis=1,
    )
)

profiles_shoreline_intersections = gpd.GeoDataFrame(
    profiles_shoreline_intersections[
        ["profile_id"] + columns_to_keep + ["shoreline_position_m", "geometry"]
    ],
    geometry="geometry",
    crs=profiles.crs,
)

# Create a single column for the date and time of the shoreline measurement
profiles_shoreline_intersections["datetime"] = pd.to_datetime(
    profiles_shoreline_intersections["Fecha"].astype(str)
    + " "
    + profiles_shoreline_intersections["Hora"].astype(str),
    format="%Y%m%d %H%M",
    errors="coerce",
)

Point intersections: 589806
